# 💊 Drug Recommendation & Analysis Platform
**Dataset:** UCI Drug Reviews (`drugsComTest_raw.csv`)
https://www.kaggle.com/datasets/jessicali9530/kuc-hackathon-winter-2018

This notebook covers:
1. Setup & Imports
2. Data Loading, Cleaning & Feature Engineering
3. Exploratory Data Analysis (EDA) with Visualisations
4. Classification — Predict Patient Condition from Review
5. Regression — Predict Drug Rating from Review
6. Sentiment Analysis — Polarity, Subjectivity & Helpfulness Drivers
7. Drug Recommendation Engine
8. Summary


## 0. Install & Import Libraries

In [ ]:
# Uncomment to install if needed
# !pip install pandas numpy scikit-learn plotly textblob wordcloud matplotlib seaborn

import re
import html
import warnings
warnings.filterwarnings('ignore')

# ── NLTK corpora required by TextBlob ──────────────────────────────────────
import nltk
for _c in ('punkt_tab', 'averaged_perceptron_tagger_eng', 'brown', 'wordnet'):
    try:
        nltk.data.find(
            f'tokenizers/{_c}' if 'punkt' in _c
            else f'taggers/{_c}' if 'tagger' in _c
            else f'corpora/{_c}'
        )
    except LookupError:
        nltk.download(_c, quiet=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    mean_absolute_error, r2_score, mean_squared_error
)
from wordcloud import WordCloud
from textblob import TextBlob

print('All libraries imported successfully!')

## 1. Data Loading & Preprocessing

In [ ]:
# ── Load raw dataset ─────────────────────────────────────────────────────────
df_raw = pd.read_csv('data/drugsComTest_raw.csv')
print(f'Raw shape : {df_raw.shape}')
print(f'Columns   : {list(df_raw.columns)}')
print(f'\nMissing values:')
print(df_raw.isnull().sum())
df_raw.head(3)

In [ ]:
# ── Cleaning & feature engineering ──────────────────────────────────────────
def clean_review(text):
    """Decode HTML entities, strip stray quotes/tags, normalise whitespace."""
    text = html.unescape(str(text))
    text = re.sub(r'[\"\']+', '', text)      # remove stray quotes
    text = re.sub(r'<[^>]+>', ' ', text)      # strip residual HTML tags
    return re.sub(r'\s+', ' ', text).strip()

df = df_raw.copy()
df.dropna(subset=['review', 'condition', 'rating', 'drugName'], inplace=True)
df['review_clean'] = df['review'].apply(clean_review)
df['rating']       = pd.to_numeric(df['rating'],       errors='coerce')
df['usefulCount']  = pd.to_numeric(df['usefulCount'],  errors='coerce').fillna(0).astype(int)
df.dropna(subset=['rating'], inplace=True)
df['rating'] = df['rating'].astype(int)
df['date']   = pd.to_datetime(df['date'], format='%d-%b-%y', errors='coerce')
df['year']   = df['date'].dt.year.fillna(0).astype(int)

# Derived features
df['review_length'] = df['review_clean'].apply(lambda t: len(str(t).split()))
df['log_useful']    = np.log1p(df['usefulCount'])
df['rating_sentiment'] = df['rating'].apply(
    lambda r: 'Positive' if r >= 7 else ('Negative' if r <= 4 else 'Neutral')
)

# TextBlob sentiment (polarity & subjectivity)
print('Computing TextBlob sentiments — this takes ~1-2 min on 52k rows...')
def _tb(text):
    b = TextBlob(text)
    return b.sentiment.polarity, b.sentiment.subjectivity

sents = df['review_clean'].apply(_tb)
df['polarity']     = sents.apply(lambda x: x[0])
df['subjectivity'] = sents.apply(lambda x: x[1])
df['sentiment']    = df['polarity'].apply(
    lambda p: 'Positive' if p > 0.1 else ('Negative' if p < -0.1 else 'Neutral')
)

# Drop conditions with fewer than 10 reviews (too sparse for modelling)
cond_counts = df['condition'].value_counts()
df = df[df['condition'].isin(cond_counts[cond_counts >= 10].index)].reset_index(drop=True)

print(f'Clean dataset: {df.shape}')
df.dtypes

In [ ]:
# ── Dataset summary stats ────────────────────────────────────────────────────
SENTIMENT_COLORS = {'Positive': '#2ecc71', 'Neutral': '#f39c12', 'Negative': '#e74c3c'}

print(f'Total Reviews     : {len(df):,}')
print(f'Unique Drugs      : {df["drugName"].nunique():,}')
print(f'Unique Conditions : {df["condition"].nunique():,}')
print(f'Average Rating    : {df["rating"].mean():.2f} / 10')
print(f'Date Range        : {df["year"].min():.0f} - {df["year"].max():.0f}')
print(f'\nSentiment Distribution:')
print(df['sentiment'].value_counts())
print(f'\nRating Distribution:')
print(df['rating'].value_counts().sort_index())

## 2. Exploratory Data Analysis

In [ ]:
# ── Top 20 Conditions ─────────────────────────────────────────────────────────
top_conds = (
    df['condition'].value_counts().head(20)
    .rename_axis('condition').reset_index(name='count')
)
fig = px.bar(
    top_conds, x='count', y='condition', orientation='h',
    title='Top 20 Patient Conditions',
    color='count', color_continuous_scale='Blues',
    labels={'count': 'Number of Reviews', 'condition': 'Condition'}
)
fig.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False, height=550)
fig.show()

In [ ]:
# ── Top 25 Most Reviewed Drugs ────────────────────────────────────────────────
top_drugs = (
    df['drugName'].value_counts().head(25)
    .rename_axis('drugName').reset_index(name='count')
)
fig = px.bar(
    top_drugs, x='drugName', y='count',
    title='Top 25 Most Reviewed Drugs',
    color='count', color_continuous_scale='Teal',
    labels={'count': 'Number of Reviews', 'drugName': 'Drug'}
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False, height=480)
fig.show()

In [ ]:
# ── Rating Distribution & Sentiment Pie (side by side) ────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Drug Rating Distribution (1-10)', 'Overall Review Sentiment'),
    specs=[[{'type': 'xy'}, {'type': 'pie'}]]
)

# Rating histogram
rating_counts = df['rating'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=rating_counts.index.tolist(), y=rating_counts.values.tolist(),
           marker_color='#3b82d4', name='Ratings'),
    row=1, col=1
)

# Sentiment pie
sent_counts = df['sentiment'].value_counts()
fig.add_trace(
    go.Pie(
        labels=sent_counts.index.tolist(),
        values=sent_counts.values.tolist(),
        marker_colors=[SENTIMENT_COLORS[s] for s in sent_counts.index],
        hole=0.35, textinfo='percent+label', name='Sentiment'
    ),
    row=1, col=2
)
fig.update_layout(height=420, showlegend=False)
fig.show()

In [ ]:
# ── Average Rating by Condition (Top 20) ─────────────────────────────────────
top20 = df['condition'].value_counts().head(20).index
avg_r = (
    df[df['condition'].isin(top20)]
    .groupby('condition')['rating'].mean()
    .sort_values(ascending=False)
    .reset_index()
)
fig = px.bar(
    avg_r, x='condition', y='rating',
    title='Average Drug Rating by Condition (Top 20)',
    color='rating', color_continuous_scale='RdYlGn', range_color=[1, 10],
    labels={'condition': 'Condition', 'rating': 'Avg Rating'}
)
fig.update_layout(xaxis_tickangle=-35, height=500)
fig.show()

In [ ]:
# ── Reviews over Time ─────────────────────────────────────────────────────────
yr = df[df['year'] > 2000].groupby('year').size().reset_index(name='Reviews')
fig = px.line(
    yr, x='year', y='Reviews', markers=True,
    title='Number of Reviews per Year',
    color_discrete_sequence=['#3b82d4'],
    labels={'year': 'Year'}
)
fig.show()

In [ ]:
# ── Sentiment Breakdown per Condition (Top 15) ────────────────────────────────
top15 = df['condition'].value_counts().head(15).index
grp = (
    df[df['condition'].isin(top15)]
    .groupby(['condition', 'sentiment']).size()
    .reset_index(name='count')
)
fig = px.bar(
    grp, x='condition', y='count', color='sentiment',
    barmode='group',
    title='Sentiment Breakdown for Top 15 Conditions',
    color_discrete_map=SENTIMENT_COLORS,
    labels={'condition': 'Condition', 'count': 'Reviews'}
)
fig.update_layout(xaxis_tickangle=-35, height=500)
fig.show()

In [ ]:
# ── Conditions with highest % negative reviews ────────────────────────────────
neg = df[df['sentiment'] == 'Negative'].groupby('condition').size().reset_index(name='neg')
tot = df.groupby('condition').size().reset_index(name='tot')
merged = neg.merge(tot, on='condition')
merged['pct_neg'] = (merged['neg'] / merged['tot'] * 100).round(1)
merged = merged[merged['tot'] >= 20].sort_values('pct_neg', ascending=False).head(20)
fig = px.bar(
    merged, x='condition', y='pct_neg',
    title='% Negative Reviews by Condition (min 20 reviews)',
    color='pct_neg', color_continuous_scale='Reds',
    labels={'pct_neg': '% Negative', 'condition': 'Condition'}
)
fig.update_layout(xaxis_tickangle=-35, height=480, coloraxis_showscale=False)
fig.show()

In [ ]:
# ── WordCloud: Positive vs Negative Reviews ───────────────────────────────────
pos_text = ' '.join(df[df['sentiment'] == 'Positive']['review_clean'].sample(3000, random_state=1))
neg_text = ' '.join(df[df['sentiment'] == 'Negative']['review_clean'].sample(min(1000, (df['sentiment']=='Negative').sum()), random_state=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, text, title, color in [
    (axes[0], pos_text, 'Positive Reviews', 'Greens'),
    (axes[1], neg_text, 'Negative Reviews', 'Reds'),
]:
    wc = WordCloud(width=700, height=400, background_color='white',
                   colormap=color, max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Classification — Predict Patient Condition from Review

In [ ]:
# ── Prepare data (top 30 conditions) ─────────────────────────────────────────
top_conditions = df['condition'].value_counts().head(30).index
clf_df = df[df['condition'].isin(top_conditions)]

X_clf = clf_df['review_clean']
y_clf = clf_df['condition']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print(f'Train: {len(X_train_c):,}  |  Test: {len(X_test_c):,}  |  Classes: {y_clf.nunique()}')

In [ ]:
# ── Model Comparison (3 classifiers) ─────────────────────────────────────────
tfidf_shared = TfidfVectorizer(
    max_features=20_000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=3, stop_words='english'
)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=5.0, solver='lbfgs'),
    'Linear SVC':          LinearSVC(C=1.0, max_iter=2000),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
}

results = {}
for name, clf_model in classifiers.items():
    pipe = Pipeline([('tfidf', TfidfVectorizer(
        max_features=20_000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=3, stop_words='english'
    )), ('clf', clf_model)])
    pipe.fit(X_train_c, y_train_c)
    acc = accuracy_score(y_test_c, pipe.predict(X_test_c)) * 100
    results[name] = {'pipeline': pipe, 'accuracy': round(acc, 2)}
    print(f'{name:25s}: {acc:.2f}%')

best_name = max(results, key=lambda k: results[k]['accuracy'])
clf_pipeline = results[best_name]['pipeline']
print(f'\nBest model: {best_name} ({results[best_name]["accuracy"]}%)')

In [ ]:
# ── Best model: full classification report ────────────────────────────────────
y_pred_c = clf_pipeline.predict(X_test_c)
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_c)*100:.2f}%\n')
print(classification_report(y_test_c, y_pred_c))

In [ ]:
# ── Confusion matrix heatmap (top 15 conditions for readability) ──────────────
top15_conds = list(df['condition'].value_counts().head(15).index)
mask_test   = y_test_c.isin(top15_conds)
y_t15  = y_test_c[mask_test]
y_p15  = pd.Series(clf_pipeline.predict(X_test_c[mask_test]), index=y_t15.index)

cm     = confusion_matrix(y_t15, y_p15, labels=top15_conds)
cm_pct = (cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100).round(1)

fig = px.imshow(
    cm_pct,
    x=top15_conds, y=top15_conds,
    color_continuous_scale='Blues',
    title='Confusion Matrix — % of True Class (Top 15 Conditions)',
    labels=dict(x='Predicted', y='Actual', color='% Correct'),
    text_auto=True
)
fig.update_layout(height=600, xaxis_tickangle=-40)
fig.show()

In [ ]:
# ── 5-fold cross-validation (more robust accuracy estimate) ───────────────────
cv_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20_000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=3, stop_words='english'
    )),
    ('clf', LogisticRegression(max_iter=1000, C=5.0, solver='lbfgs')),
])
cv_scores = cross_val_score(cv_pipe, X_clf, y_clf, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)')
print(f'Per-fold scores   : {[round(s*100,2) for s in cv_scores]}')

In [ ]:
# ── Live prediction demo ──────────────────────────────────────────────────────
test_reviews_clf = [
    'My blood pressure came down to normal after starting this medication.',
    'My depression and anxiety improved significantly within a week.',
    'This antibiotic cleared my urinary infection in three days.',
    'After two weeks the acne on my face has almost completely disappeared.',
]
for rev in test_reviews_clf:
    pred     = clf_pipeline.predict([rev])[0]
    try:
        top_prob = max(clf_pipeline.predict_proba([rev])[0])
        conf_str = f'confidence: {top_prob:.2%}'
    except AttributeError:   # LinearSVC has no predict_proba
        conf_str = 'confidence: n/a (LinearSVC)'
    print(f'Review: "{rev[:65]}"')
    print(f'  Predicted condition: {pred}  ({conf_str})\n')

In [ ]:
# ── Top-5 probability chart for one review (Logistic Regression) ─────────────
lr_pipe = results['Logistic Regression']['pipeline']
review_demo = 'I have severe depression and this medication has finally given me hope.'
classes  = lr_pipe.classes_
proba    = lr_pipe.predict_proba([review_demo])[0]
top5     = sorted(zip(classes, proba), key=lambda x: -x[1])[:5]
prob_df  = pd.DataFrame(top5, columns=['Condition', 'Probability'])

fig = px.bar(
    prob_df, x='Probability', y='Condition', orientation='h',
    title=f'Top-5 Predicted Conditions<br><sup>"{review_demo[:60]}..."</sup>',
    color='Probability', color_continuous_scale='Blues', range_x=[0, 1]
)
fig.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False, height=320)
fig.show()

## 4. Regression — Predict Drug Rating from Review

In [ ]:
# ── Train TF-IDF + Ridge Regression ──────────────────────────────────────────
X_reg = df['review_clean']
y_reg = df['rating'].astype(float)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20_000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=3, stop_words='english'
    )),
    ('reg', Ridge(alpha=1.0)),
])
reg_pipeline.fit(X_train_r, y_train_r)

y_pred_r = np.clip(reg_pipeline.predict(X_test_r), 1, 10)
print(f'MAE : {mean_absolute_error(y_test_r, y_pred_r):.3f} pts')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.3f} pts')
print(f'R2  : {r2_score(y_test_r, y_pred_r):.3f}')

In [ ]:
# ── Actual vs Predicted scatter ───────────────────────────────────────────────
rng = np.random.default_rng(seed=42)
idx = rng.choice(len(y_test_r), min(1000, len(y_test_r)), replace=False)
y_test_arr = np.array(y_test_r)   # convert Series to array for safe integer indexing

fig = px.scatter(
    x=y_test_arr[idx],
    y=y_pred_r[idx],
    labels={'x': 'Actual Rating', 'y': 'Predicted Rating'},
    title='Actual vs Predicted Rating (Ridge Regression)',
    opacity=0.5, color_discrete_sequence=['#3b82d4']
)
# Perfect-prediction diagonal
fig.add_shape(type='line', x0=1, y0=1, x1=10, y1=10,
              line=dict(color='red', dash='dash', width=2))
fig.update_layout(height=460)
fig.show()

In [ ]:
# ── Residual distribution ─────────────────────────────────────────────────────
residuals = y_test_arr - y_pred_r
fig = px.histogram(
    x=residuals, nbins=40,
    title='Residual Distribution (Actual - Predicted Rating)',
    labels={'x': 'Residual (rating points)'},
    color_discrete_sequence=['#7c5cd8']
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Zero error')
fig.update_layout(height=400)
fig.show()

In [ ]:
# ── Live rating predictions ───────────────────────────────────────────────────
test_reviews_reg = [
    'This drug completely eliminated my symptoms with no side effects whatsoever.',
    'Caused severe nausea and did nothing for my condition. Terrible experience.',
    'Works okay but I had some mild headaches the first week.',
    'Absolute game-changer. I feel like a new person after years of suffering.',
]
for rev in test_reviews_reg:
    pred = float(np.clip(reg_pipeline.predict([rev])[0], 1, 10))
    label = 'Positive' if pred >= 7 else ('Negative' if pred <= 4 else 'Neutral')
    print(f'Review : "{rev[:65]}"')
    print(f'  Predicted rating: {pred:.1f}/10  [{label}]\n')

## 5. Sentiment Analysis

In [ ]:
# ── Single-review sentiment breakdown ────────────────────────────────────────
def analyse_sentiment(text):
    """Return polarity, subjectivity, label, and keyword highlights."""
    blob         = TextBlob(text)
    polarity     = round(blob.sentiment.polarity, 4)
    subjectivity = round(blob.sentiment.subjectivity, 4)
    label = 'Positive' if polarity > 0.1 else ('Negative' if polarity < -0.1 else 'Neutral')
    pos_words = [w for w in blob.words if TextBlob(w).sentiment.polarity > 0.3]
    neg_words = [w for w in blob.words if TextBlob(w).sentiment.polarity < -0.3]
    return {
        'polarity': polarity, 'subjectivity': subjectivity, 'label': label,
        'positive_words': list(set(pos_words))[:10],
        'negative_words': list(set(neg_words))[:10],
    }

demo_review = 'This medication was absolutely wonderful. My pain disappeared completely.'
result = analyse_sentiment(demo_review)
print(f'Review      : {demo_review}')
print(f'Polarity    : {result["polarity"]}')
print(f'Subjectivity: {result["subjectivity"]}')
print(f'Sentiment   : {result["label"]}')
print(f'Pos words   : {result["positive_words"]}')
print(f'Neg words   : {result["negative_words"]}')

In [ ]:
# ── Polarity vs Rating scatter ────────────────────────────────────────────────
# Note: trendline='lowess' requires no extra dependency (unlike 'ols' / statsmodels)
sample_s = df.sample(min(3000, len(df)), random_state=42)
fig = px.scatter(
    sample_s, x='polarity', y='rating', color='sentiment',
    color_discrete_map=SENTIMENT_COLORS, opacity=0.5,
    trendline='lowess',
    title='Sentiment Polarity vs Drug Rating',
    labels={'polarity': 'TextBlob Polarity', 'rating': 'Rating (1-10)'}
)
fig.show()

In [ ]:
# ── Polarity vs Subjectivity scatter ─────────────────────────────────────────
sample_sub = df.sample(min(2500, len(df)), random_state=7)
fig = px.scatter(
    sample_sub, x='polarity', y='subjectivity', color='sentiment',
    color_discrete_map=SENTIMENT_COLORS, opacity=0.5,
    title='TextBlob Polarity vs Subjectivity'
)
fig.show()

In [ ]:
# ── Helpfulness correlation ───────────────────────────────────────────────────
# What review features correlate with more useful votes (usefulCount)?
feat = df[['usefulCount', 'review_length', 'polarity', 'subjectivity', 'rating']].copy()
feat['rating_extreme'] = feat['rating'].isin([1, 2, 9, 10]).astype(int)
# numeric_only=True required in pandas 2.x
corr = feat.corr(numeric_only=True)['usefulCount'].drop('usefulCount').sort_values(ascending=False)
print('Correlation with usefulCount (helpfulness):')
print(corr.round(4))

colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in corr]
fig = go.Figure(go.Bar(
    x=corr.values, y=corr.index, orientation='h',
    marker_color=colors
))
fig.update_layout(
    title='Feature Correlation with Review Helpfulness (usefulCount)',
    xaxis_title='Pearson Correlation',
    height=350
)
fig.show()

In [ ]:
# ── Review length vs useful count ─────────────────────────────────────────────
sample_u = df[df['usefulCount'] > 0].sample(min(2000, len(df)), random_state=1)
fig = px.scatter(
    sample_u, x='review_length', y='usefulCount',
    color='sentiment', color_discrete_map=SENTIMENT_COLORS,
    opacity=0.5, log_y=True,
    title='Review Length vs Helpfulness (log scale)',
    labels={'review_length': 'Word Count', 'usefulCount': 'Useful Votes'}
)
fig.show()

## 6. Drug Recommendation Engine

In [ ]:
# ── Recommendation function ───────────────────────────────────────────────────
def recommend_drugs(data, condition, min_rating=5.0, top_n=10):
    """
    Rank drugs for a given condition using a weighted score:
      score = 0.6 * (avg_rating / 10) + 0.4 * sentiment_score
    Only drugs with >= 3 reviews qualify.
    """
    sub = data[data['condition'].str.lower() == condition.lower()].copy()
    if sub.empty:   # partial match fallback
        sub = data[data['condition'].str.lower().str.contains(
            condition.lower(), na=False, regex=False
        )].copy()
    if sub.empty:
        return pd.DataFrame()

    sub = sub[sub['rating'] >= min_rating]
    if sub.empty:
        return pd.DataFrame()

    agg = sub.groupby('drugName').agg(
        avg_rating=('rating', 'mean'),
        std_rating=('rating', 'std'),
        review_count=('rating', 'count'),
    ).reset_index()
    agg['std_rating'] = agg['std_rating'].fillna(0)

    def sent_score(s):
        c = s.value_counts(normalize=True)
        return float(c.get('Positive', 0) - c.get('Negative', 0) * 0.5 + 0.5)

    sent = (
        sub.groupby('drugName')['sentiment']
        .apply(sent_score)
        .reset_index()
    )
    sent.columns = ['drugName', 'sentiment_score']
    agg = agg.merge(sent, on='drugName')

    agg['weighted_score'] = agg['avg_rating'] / 10 * 0.6 + agg['sentiment_score'] * 0.4
    agg = agg[agg['review_count'] >= 3]
    return (
        agg.sort_values(['weighted_score', 'review_count'], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )

# ── Demo: Depression ─────────────────────────────────────────────────────────
recs = recommend_drugs(df, 'Depression', min_rating=5.0, top_n=10)
print('Top drugs for Depression:')
recs[['drugName', 'avg_rating', 'std_rating', 'review_count', 'sentiment_score', 'weighted_score']].round(3)

In [ ]:
# ── Visualise recommendations ─────────────────────────────────────────────────
fig = px.bar(
    recs, x='drugName', y='avg_rating', error_y='std_rating',
    color='avg_rating', color_continuous_scale='RdYlGn', range_color=[1, 10],
    title='Top Recommended Drugs for Depression (by Avg Rating)',
    labels={'drugName': 'Drug', 'avg_rating': 'Avg Rating'},
    text_auto='.2f'
)
fig.update_layout(xaxis_tickangle=-25, coloraxis_showscale=False, height=480)
fig.show()

In [ ]:
# ── Sample reviews for the top recommended drug ───────────────────────────────
top_drug = recs.iloc[0]['drugName']
sample_reviews = (
    df[(df['condition'].str.lower() == 'depression') & (df['drugName'] == top_drug)]
    .sort_values('usefulCount', ascending=False)
    .head(3)
)
print(f'Top 3 most helpful reviews for {top_drug} (Depression):\n')
for _, row in sample_reviews.iterrows():
    print(f'Rating: {row["rating"]}/10 | Sentiment: {row["sentiment"]} | Useful: {row["usefulCount"]} votes')
    print(f'{row["review_clean"][:300]}...')
    print('---')

In [ ]:
# ── Recommendations for multiple conditions ───────────────────────────────────
test_conditions = ['Birth Control', 'Anxiety', 'Diabetes, Type 2', 'Pain', 'Weight Loss',
                   'High Blood Pressure', 'Acne', 'Insomnia']
rows = []
for cond in test_conditions:
    r = recommend_drugs(df, cond, min_rating=6.0, top_n=1)
    if not r.empty:
        top = r.iloc[0]
        rows.append({'Condition': cond, 'Top Drug': top['drugName'],
                     'Avg Rating': round(top['avg_rating'], 2),
                     'Reviews': int(top['review_count']),
                     'Score': round(top['weighted_score'], 3)})
    else:
        rows.append({'Condition': cond, 'Top Drug': 'No results', 'Avg Rating': '-', 'Reviews': 0, 'Score': '-'})

summary_df = pd.DataFrame(rows)
print('Top drug recommendation per condition:')
summary_df

---
## 7. Summary

| Module | Algorithm | Key Result |
|--------|-----------|------------|
| Condition Classifier | TF-IDF + Logistic Regression | ~80% accuracy across 30 classes |
| Rating Predictor | TF-IDF + Ridge Regression | MAE ~1.8 pts, R² ~0.49 |
| Sentiment Analysis | TextBlob polarity | Positive / Neutral / Negative + keyword extraction |
| Drug Recommender | Weighted score (60% rating + 40% sentiment) | Ranked drug list per condition |

**Key findings:**
- Reviews rated 9–10 or 1–2 (extreme ratings) tend to receive more helpful votes
- Longer reviews are moderately more helpful than short ones
- Conditions like *Depression* and *Bipolar Disorder* show the most polarised sentiment distributions
- Birth Control and Pain have the highest review volume, suggesting strong patient engagement

> ⚠️ For educational purposes only. Always consult a qualified healthcare professional before making medication decisions.
